In [19]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from transformers import (
    BertTokenizer,
    BertForMaskedLM,
    BertConfig,
    LineByLineTextDataset,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
import torch
from sklearn.model_selection import train_test_split


In [20]:
nbs_ = pd.read_csv('E:\\GitHub\\Multi-Model-Bias-Detection-and-Debiasing-the-News\\EDA\\No_Corrupted_NBS.csv')

In [21]:
nbs_

,unique_id,outlet_x,title,content,image_description_x,text_label_x,multimodal_label_x,outlet_y,headline,top_image,article_text,new_categories,news_categories_confidence_scores,text_label_y,multimodal_label_y,image_description_y,image_filename,image_path
0,23400064a2,CBC.ca,Anishnabeg Outreach develops self-serve mental...,A new resource developed by Anishnabeg Outreac...,The image shows a person standing in front of ...,Unlikely,Likely,CBC.ca,Anishnabeg Outreach develops self-serve mental...,"{'bytes': None, 'path': '/projects/NMB-Plus/ra...",A new resource developed by Anishnabeg Outreac...,['Health' 'Local/Regional'],[0.85 0.75],Unlikely,Likely,The image shows a person standing in front of ...,23400064a2.jpeg,C:\Users\ritik\Downloads\images\images\2340006...
1,0bad329c25,CBC.ca,South Asian newcomers to Canada say online hat...,International student Miran Kadri had many thi...,Two individuals with backpacks walking on a ci...,Likely,Unlikely,CBC.ca,South Asian newcomers to Canada say online hat...,"{'bytes': None, 'path': '/projects/NMB-Plus/ra...",International student Miran Kadri had many thi...,['National' 'Opinion/Editorial'],[0.85 0.65],Likely,Unlikely,Two individuals with backpacks walking on a ci...,0bad329c25.jpeg,C:\Users\ritik\Downloads\images\images\0bad329...
2,267f8bc361,CBC.ca,Program that pairs nurses with RCMP on mental ...,A program in Fort McMurray that pairs police o...,A smiling woman in a pink shirt is seated in a...,Unlikely,Likely,CBC.ca,Program that pairs nurses with RCMP on mental ...,"{'bytes': None, 'path': '/projects/NMB-Plus/ra...",A program in Fort McMurray that pairs police o...,['Local/Regional' 'Health'],[0.85 0.75],Unlikely,Likely,A smiling woman in a pink shirt is seated in a...,267f8bc361.jpeg,C:\Users\ritik\Downloads\images\images\267f8bc...
3,2fab02aa42,CBC.ca,Should social media come with a health warning...,The U.S. surgeon general has called for social...,"A person is holding a smartphone, possibly rea...",Likely,Unlikely,CBC.ca,Should social media come with a health warning...,"{'bytes': None, 'path': '/projects/NMB-Plus/ra...",The U.S. surgeon general has called for social...,['Health' 'Opinion/Editorial'],[0.9 0.7],Likely,Unlikely,"A person is holding a smartphone, possibly rea...",2fab02aa42.jpeg,C:\Users\ritik\Downloads\images\images\2fab02a...
4,004798d706,CBC.ca,Inside Out 2: What the movie’s 4 new emotions ...,"Emotions are clues about ourselves, says exper...",A character with blue hair and a green dress s...,Likely,Unlikely,CBC.ca,Inside Out 2: What the movie’s 4 new emotions ...,"{'bytes': None, 'path': '/projects/NMB-Plus/ra...","Emotions are clues about ourselves, says exper...",['Entertainment' 'Health'],[0.85 0.75],Likely,Unlikely,A character with blue hair and a green dress s...,004798d706.jpeg,C:\Users\ritik\Downloads\images\images\004798d...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6244,0471e108dd,CNN,Opinion: We Germans are making Trump ‘thunders...,Editor’s Note: Anna Sauerbrey is the foreign e...,A black and white photo of a large assembly ha...,Likely,Likely,CNN,Opinion: We Germans are making Trump ‘thunders...,"{'bytes': None, 'path': '/projects/NMB-Plus/wh...",Editor’s Note: Anna Sauerbrey is the foreign e...,['Opinion/Editorial' 'Politics'],[0.9 0.8],Likely,Likely,A black and white photo of a large assembly ha...,0471e108dd.jpg,C:\Users\ritik\Downloads\images\images\0471e10...
6245,1e9e145f6e,CNN,Brazil’s floods smashed through barriers desig...,Karine Pitana had just paid off the last insta...,An aerial view of a flooded road with a yellow...,Likely,Likely,CNN,Brazil’s floods smashed through barriers desig...,"{'bytes': None, 'path': '/projects/NMB-Plus/wh...",Karine Pitana had just paid off the last insta...,['Local/Regional' 'National'],[0.85 0.75],Likely,Likely,An aerial view of a flooded road with a yellow...,1e9e145f6e.jpg,C:\Users\ritik\Downloads\images\images\1e9e145...
6246,0bc2d5dd25,CNN,"In a city cut off from the w

In [28]:
nbs_['text_labels']=nbs_['text_label_x'].apply(lambda x : 'biased' if x=='Likely' else 'non-biased')

In [29]:
nbs_['MultiModal_Label'] =nbs_['multimodal_label_x']
nbs_=nbs_.drop(columns=['multimodal_label_x','multimodal_label_y'])

In [30]:
nbs_['caption']=nbs_['image_description_x']
nbs_=nbs_.drop(columns=['image_description_x','image_description_y'])

In [31]:
nbs_['MultiModal_Label']=nbs_['MultiModal_Label'].apply(lambda x : 'biased' if x=='Likely' else 'non-biased')

In [32]:
X_train, X_test = train_test_split(nbs_.iloc[5000:], test_size=0.3, random_state=42, stratify=nbs_.iloc[5000:]['MultiModal_Label'])


NBS plus Model that is text + image

Defining the model 

In [ ]:
import torch
from torch import nn
from transformers import AutoModel
from huggingface_hub import hf_hub_download
from typing import Literal
import json

class MultimodalClassifier(nn.Module):
    def __init__(
            self,
            text_encoder_id_or_path: str,
            image_encoder_id_or_path: str,
            projection_dim: int,
            fusion_method: Literal["concat", "align", "cosine_similarity"] = "concat",
            proj_dropout: float = 0.1,
            fusion_dropout: float = 0.1,
            num_classes: int = 2,
        ) -> None:
        super().__init__()

        self.fusion_method = fusion_method
        self.projection_dim = projection_dim
        self.num_classes = num_classes

        ##### Text Encoder
        self.text_encoder = AutoModel.from_pretrained(text_encoder_id_or_path)
        self.text_projection = nn.Sequential(
            nn.Linear(self.text_encoder.config.hidden_size, self.projection_dim),
            nn.Dropout(proj_dropout),
        )

        ##### Image Encoder (using ResNet34 from AutoModel with timm)
        self.image_encoder = AutoModel.from_pretrained(image_encoder_id_or_path, trust_remote_code=True)
        self.image_encoder.classifier = nn.Identity()  # rm the classification head
        self.image_projection = nn.Sequential(
            nn.Linear(512, self.projection_dim),
            nn.Dropout(proj_dropout),
        )

        ##### Fusion Layer
        fusion_input_dim = self.projection_dim * 2 if fusion_method == "concat" else self.projection_dim
        self.fusion_layer = nn.Sequential(
            nn.Dropout(fusion_dropout),
            nn.Linear(fusion_input_dim, self.projection_dim),
            nn.GELU(),
            nn.Dropout(fusion_dropout),
        )

        ##### Classification Layer
        self.classifier = nn.Linear(self.projection_dim, self.num_classes)

    def forward(self, pixel_values: torch.Tensor, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        ##### Text Encoder Projection #####
        full_text_features = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True).last_hidden_state
        full_text_features = full_text_features[:, 0, :]  # using cls token
        full_text_features = self.text_projection(full_text_features)

        ##### Image Encoder Projection #####
        resnet_image_features = self.image_encoder(pixel_values=pixel_values).last_hidden_state
        
        # global average pooling for resent image features (bad idea? dim problems)
        resnet_image_features = resnet_image_features.mean(dim=[-2, -1])
        resnet_image_features = self.image_projection(resnet_image_features)

        ##### Fusion and Classification #####
        if self.fusion_method == "concat":
            fused_features = torch.cat([full_text_features, resnet_image_features], dim=-1)
        else:
            fused_features = full_text_features * resnet_image_features # don't think this works atm (should be dot prod)

        # fusion and classifier layers
        fused_features = self.fusion_layer(fused_features)
        classification_output = self.classifier(fused_features)

        return classification_output

def load_model():
    config_path = hf_hub_download(repo_id="maximuspowers/multimodal-bias-classifier", filename="config.json")
    with open(config_path, "r") as f:
        config = json.load(f)

    model = MultimodalClassifier(
        text_encoder_id_or_path=config["text_encoder_id_or_path"],
        image_encoder_id_or_path="microsoft/resnet-34",
        projection_dim=config["projection_dim"],
        fusion_method=config["fusion_method"],
        proj_dropout=config["proj_dropout"],
        fusion_dropout=config["fusion_dropout"],
        num_classes=config["num_classes"]
    )

    model_weights_path = hf_hub_download(repo_id="maximuspowers/multimodal-bias-classifier", filename="model_weights.pth")
    checkpoint = torch.load(model_weights_path, map_location=torch.device('cpu'))
    model.load_state_dict(checkpoint, strict=False)

    return model


Loading the saved model

In [ ]:
dict=torch.load("fine_tuned_model_nbs.pt")
model = load_model()
model.load_state_dict(dict)
model.eval()

MultimodalClassifier(
  (text_encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-

Text Based Model (BABE Model)

Defining the based Model

In [ ]:
from transformers import BertModel

class BertClass(torch.nn.Module):
    def __init__(self):
        super(BertClass, self).__init__()
        self.l1 = BertModel.from_pretrained("E:\\GitHub\\Multi-Model-Bias-Detection-and-Debiasing-the-News\\EDA\\Model_config")
        self.pre_classifier = torch.nn.Linear(768, 768)
        self.dropout = torch.nn.Dropout(0.1)
        self.classifier = torch.nn.Linear(768, 2)
        self.relu = torch.nn.ReLU()

    def forward(self, input_ids, attention_mask, token_type_ids):
        output_1 = self.l1(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        hidden_state = output_1[0]
        pooler = hidden_state[:, 0]
        pooler = self.pre_classifier(pooler)
        pooler = self.relu(pooler)
        pooler = self.dropout(pooler)
        output = self.classifier(pooler)
        return output


Loading the pre trained weights

In [ ]:
dict=torch.load("BABE_fine_tuned_model.pt")
model = BertClass()
model.load_state_dict(dict)
model.eval()

Some weights of BertModel were not initialized from the model checkpoint at E:\GitHub\Multi-Model-Bias-Detection-and-Debiasing-the-News\EDA\Model_config and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertClass(
  (l1): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-5): 6 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=

In [ ]:
import torch
from transformers import AutoTokenizer
from PIL import Image
import requests
from torchvision import transforms

Tokenizer

In [ ]:
text_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

Babe Dataset Data Loader and Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch
class QueryData(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.text1 = dataframe['article'].values
        self.text2 = dataframe['text'].values
        self.targets = dataframe['label_bias'].values
        self.max_len = max_len

    def __len__(self):
        return len(self.text1)

    def __getitem__(self, index):
        text1 = str(self.text1[index])
        text2 = str(self.text2[index])

        inputs = self.tokenizer(

            text1,
            text2,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_token_type_ids=True,
            return_attention_mask=True
        )
        ids = inputs['input_ids']
        mask = inputs['attention_mask']
        token_type_ids = inputs["token_type_ids"]


        return {
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long),
            'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long),
            'targets': torch.tensor(self.targets[index], dtype=torch.float)
        }
    
training_set = QueryData(X_train, tokenizer, MAX_LEN)
val_set = QueryData(X_test, tokenizer, MAX_LEN)

TRAIN_BATCH_SIZE=8
VALID_BATCH_SIZE=8
# Defining training and testing paramerters
train_params = {'batch_size': TRAIN_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

test_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

# Creating dataloader for training and testing purposes
training_loader = DataLoader(training_set, **train_params)
val_loader = DataLoader(val_set, **test_params)

ModuleNotFoundError: No module named 'Babe_Dataset'

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class DATA(Dataset):
    def __init__(self, dataframe, text_tokenizer, image_transform):
        self.df = dataframe
        self.text_tokenizer = text_tokenizer
        self.image_transform = image_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row['article_text']
        image_path = row['image_path']

        # tokenize text
        text_inputs = self.text_tokenizer(
            text, padding='max_length', truncation=True, max_length=512 , return_tensors='pt'
        )
        # transform image
        image = Image.open(image_path).convert('RGB')
        image_input = self.image_transform(image)

        #label = torch.tensor(row['MultiModal_Label'])
        
        input_ids= text_inputs['input_ids'][0]
        attention_mask= text_inputs['attention_mask']
        pixel_values = image_input
        labels =  int(row['MultiModal_Label'])
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'pixel_values': image_input,
            'labels': torch.tensor(labels, dtype=torch.long)
        }

# Then create dataset and loader

test_loader_dataset = DATA(X_test, text_tokenizer, image_transform)

training_loader = DataLoader(train_loader_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(test_loader_dataset, batch_size=4, shuffle=True)

In [ ]:
def valid_MultiModal(model, testing_loader):
    model.eval()
    n_correct = 0; n_wrong = 0; total = 0; tr_loss=0; nb_tr_steps=0; nb_tr_examples=0 ; True_pos=0; False_pos=0; True_neg=0; False_neg=0; n_True_pos=0;n_False_pos=0;n_True_neg=0;n_False_neg=0

    with torch.no_grad():
        for _, data in tqdm(enumerate(testing_loader, 0)):
            ids = data['input_ids'].to(device, dtype = torch.long)
            mask = data['attention_mask'].to(device, dtype = torch.long)
            pixel_values = data['pixel_values'].to(device, dtype = torch.float)
            targets = data['labels'].reshape(-1,1).to(device, dtype = torch.float)

            outputs = model(input_ids=ids,
            attention_mask=mask,
            pixel_values=pixel_values)

            loss = loss_function(outputs, targets)
            tr_loss += loss.item()
            preds = torch.sigmoid(outputs) > 0.5        
            n_correct += calcuate_accuracy(preds, targets.bool())

            n_True_pos,n_False_pos,n_True_neg,n_False_neg = calculate_truth_value(preds, targets)
            True_pos += n_True_pos
            False_pos += n_False_pos
            True_neg += n_True_neg
            False_neg += n_False_neg


            nb_tr_steps += 1
            nb_tr_examples+=targets.size(0)
            
            if _%5000==0:
                loss_step = tr_loss/nb_tr_steps
                accu_step = (n_correct*100)/nb_tr_examples

        labels = ["Biased", "Not Biased"]  

    for i in range(len(preds)):
        predicted_class = int(preds[i].item())  
        predicted_label = labels[predicted_class]
        predicted_prob = torch.sigmoid(outputs[i]).item()
        print(f"Prediction: {predicted_label}, Probability: {predicted_prob:.4f}")

    epoch_loss = tr_loss/nb_tr_steps
    epoch_accu = (n_correct*100)/nb_tr_examples
    print(f"Validation Loss Epoch: {epoch_loss}")
    print(f"Validation Accuracy Epoch: {epoch_accu}")
    
    return epoch_accu,True_pos,False_pos,True_neg,False_neg
acc,n_True_pos,n_False_pos,n_True_neg,n_False_neg = valid_MultiModal(model, val_loader)

In [ ]:
def Combined_Model(alpha):
    acc,n_True_pos,n_False_pos,n_True_neg,n_False_neg = valid(model, val_loader)
